# Notebook 08: Clustering and Dimensionality Reduction

## Why This Matters

Unsupervised learning techniques — clustering, PCA, t-SNE, UMAP — are essential for the exploratory phases of any ML project and appear frequently in interviews as feature engineering tools. PCA reduces dimensionality while preserving variance, making downstream supervised models faster and more stable; it is also the backbone of many anomaly detection and recommendation systems. Clustering algorithms (K-Means, DBSCAN, hierarchical) are used for customer segmentation, data labeling, and feature engineering (cluster membership as a feature). Understanding when each method works, what assumptions it makes, and how to evaluate quality without labels is consistently tested in ML system design interviews.

## 1. Principal Component Analysis (PCA)

PCA finds a low-dimensional linear projection that maximizes variance. Formally, it finds an orthogonal matrix W such that $Z = XW$ has maximum variance in each dimension and zero covariance between dimensions.

The solution is the **eigendecomposition of the covariance matrix** (or equivalently, SVD of the data matrix):
- Principal components = eigenvectors of $\mathbf{X}^T\mathbf{X}$, ordered by eigenvalue
- Explained variance ratio of each PC = eigenvalue / sum of eigenvalues

When to use PCA:
- Before linear models: removes multicollinearity; features become orthogonal
- Visualization: project to 2-3 components
- Compression: retain 95% of variance in far fewer dimensions
- Anomaly detection: reconstruction error in PCA space flags outliers

When NOT to use PCA:
- Tree models (scale/rotation invariant; PCA doesn't help)
- When interpretability of original features matters

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits, make_blobs, load_breast_cancer

# PCA on digits dataset (64 features -> 2D visualization)
digits = load_digits()
X_dig = StandardScaler().fit_transform(digits.data)
y_dig = digits.target

pca = PCA().fit(X_dig)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Explained variance
cumvar = np.cumsum(pca.explained_variance_ratio_)
axes[0].plot(range(1, len(cumvar)+1), cumvar, 'steelblue', marker='o', markersize=3)
axes[0].axhline(0.95, color='red', linestyle='--', label='95% variance')
k95 = np.searchsorted(cumvar, 0.95) + 1
axes[0].axvline(k95, color='gray', linestyle=':', label=f'k={k95}')
axes[0].set_xlabel('Number of components')
axes[0].set_ylabel('Cumulative explained variance')
axes[0].set_title(f'PCA: {k95} components explain 95% of variance')
axes[0].legend()

# 2D projection
X_2d = PCA(n_components=2).fit_transform(X_dig)
scatter = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=y_dig, cmap='tab10', s=8, alpha=0.7)
plt.colorbar(scatter, ax=axes[1], label='Digit')
axes[1].set_title('First 2 PCA Components — Digits Dataset')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.show()
print(f"Original dimensions: {X_dig.shape[1]}")
print(f"Dimensions for 95% variance: {k95}")

In [ ]:
# PCA as preprocessing: effect on logistic regression performance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

print("PCA as preprocessing for Logistic Regression on Digits:")
for n_comp in [2, 5, 10, 20, 30, 40, 64]:
    if n_comp == 64:
        pipe = Pipeline([('scale', StandardScaler()), ('lr', LogisticRegression(max_iter=500))])
    else:
        pipe = Pipeline([
            ('scale', StandardScaler()),
            ('pca', PCA(n_components=n_comp)),
            ('lr', LogisticRegression(max_iter=500))
        ])
    scores = cross_val_score(pipe, digits.data, y_dig, cv=5, scoring='accuracy')
    label = f'PCA({n_comp})' if n_comp < 64 else 'All 64 feats'
    print(f"  {label:15s}: accuracy = {scores.mean():.4f} ± {scores.std():.4f}")

## 2. t-SNE and UMAP for Non-Linear Dimensionality Reduction

PCA is linear — it cannot capture complex non-linear structures in the data. For visualization of high-dimensional data, two non-linear methods dominate:

**t-SNE** (t-Distributed Stochastic Neighbor Embedding):
- Converts pairwise distances to probabilities; preserves local neighborhood structure
- t-distributed (heavy-tailed) repulsion prevents crowding in 2D
- **Limitations**: non-deterministic; distances between clusters are NOT meaningful; perplexity hyperparameter matters; slow O(n²) or O(n log n) with Barnes-Hut

**UMAP** (Uniform Manifold Approximation and Projection):
- Faster than t-SNE; can handle larger datasets
- Better preserves global structure
- Can be used for dimensionality reduction (not just visualization) as a pre-processing step
- `n_neighbors` (local vs. global structure) and `min_dist` (packing) are key hyperparameters

In [ ]:
from sklearn.manifold import TSNE

# t-SNE on digits
X_subset = X_dig[:1000]  # subset for speed
y_subset = y_dig[:1000]

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=300)
X_tsne = tsne.fit_transform(X_subset)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (X_2, title) in [
    (axes[0], (PCA(n_components=2).fit_transform(X_subset), 'PCA (linear)')),
    (axes[1], (X_tsne, 't-SNE (non-linear, perplexity=30)')),
]:
    sc = ax.scatter(X_2[:, 0], X_2[:, 1], c=y_subset, cmap='tab10', s=8, alpha=0.8)
    plt.colorbar(sc, ax=ax, label='Digit')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')

plt.suptitle('PCA vs t-SNE: Digits Visualization', fontsize=12)
plt.tight_layout()
plt.show()

## 3. K-Means Clustering

K-Means partitions n samples into k clusters by iteratively:
1. Assign each point to its nearest centroid (E-step)
2. Recompute centroids as cluster means (M-step)

Converges to a local minimum of the within-cluster sum of squares (WCSS / inertia):
$$\text{WCSS} = \sum_{k=1}^K \sum_{i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$$

**Assumptions and limitations:**
- Assumes spherical, equal-size clusters — fails on elongated, irregular, or very different-sized clusters
- Sensitive to scale (use StandardScaler first)
- Sensitive to initialization — use `init='k-means++'` for smart initialization
- Must specify K in advance — use elbow method or silhouette score

**Choosing K:** The **elbow method** plots WCSS vs. K. The **silhouette score** measures how well each point fits its cluster vs. the nearest neighboring cluster (range: [-1, 1], higher is better).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Generate clustered data
X_clust, y_clust = make_blobs(n_samples=400, n_features=2, centers=4,
                               cluster_std=1.0, random_state=42)
X_clust_s = StandardScaler().fit_transform(X_clust)

# Elbow and silhouette
K_range = range(2, 11)
inertias, sil_scores = [], []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_clust_s)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_clust_s, labels))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(K_range, inertias, 'steelblue', marker='o')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method')
axes[0].axvline(4, color='red', linestyle='--', label='True K=4')
axes[0].legend()

axes[1].plot(K_range, sil_scores, 'coral', marker='s')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score (higher = better)')
axes[1].axvline(4, color='red', linestyle='--', label='True K=4')
axes[1].legend()

# Final clustering with K=4
km4 = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
labels4 = km4.fit_predict(X_clust_s)
axes[2].scatter(X_clust_s[:, 0], X_clust_s[:, 1], c=labels4, cmap='tab10', s=20, alpha=0.7)
axes[2].scatter(km4.cluster_centers_[:, 0], km4.cluster_centers_[:, 1],
                s=200, c='black', marker='X', zorder=10, label='Centroids')
axes[2].set_title('K-Means (K=4)')
axes[2].legend()

plt.tight_layout()
plt.show()

## 4. DBSCAN: Density-Based Clustering

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) discovers clusters of arbitrary shape by connecting points that are within ε distance of each other and have at least `min_samples` neighbors.

Three types of points:
- **Core point**: at least `min_samples` points within radius ε
- **Border point**: within ε of a core point but fewer than `min_samples` neighbors
- **Noise point** (label=-1): not within ε of any core point

Advantages over K-Means:
- Does not require specifying K
- Handles clusters of arbitrary shape
- Natively identifies outliers (noise points)
- Works on ring/moon/irregular cluster shapes where K-Means fails

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons

# DBSCAN vs K-Means on non-convex shapes
X_moon, y_moon = make_moons(n_samples=300, noise=0.05, random_state=42)

kmeans_moon  = KMeans(n_clusters=2, random_state=42).fit(X_moon)
dbscan_moon  = DBSCAN(eps=0.2, min_samples=5).fit(X_moon)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, labels, title in [
    (axes[0], y_moon,               'True Labels'),
    (axes[1], kmeans_moon.labels_,  'K-Means (K=2) — Fails'),
    (axes[2], dbscan_moon.labels_,  'DBSCAN (ε=0.2) — Succeeds'),
]:
    ax.scatter(X_moon[:, 0], X_moon[:, 1], c=labels, cmap='tab10', s=20, edgecolors='k', lw=0.3)
    ax.set_title(title, fontsize=10)
    n_clust = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    ax.set_xlabel(f'Clusters={n_clust}, Noise={n_noise}')

plt.suptitle('K-Means vs. DBSCAN on Non-Convex Data', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Cluster Features for Supervised Learning

Clustering results can be used as features in a supervised model — a technique called **cluster-based feature engineering**:
1. Run K-Means with k=n_clusters on training data
2. Add cluster ID as a categorical feature (encode as one-hot or target encode)
3. Optionally add distance to each centroid as numeric features

This can capture non-linear groupings that a linear model cannot express. Always fit the clusterer on training data only and transform test data separately.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin

# Custom transformer: adds cluster distances as features
class KMeansFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=8):
        self.n_clusters = n_clusters
    def fit(self, X, y=None):
        self.km_ = KMeans(n_clusters=self.n_clusters, init='k-means++', n_init=5, random_state=42)
        self.km_.fit(X)
        return self
    def transform(self, X):
        dists = self.km_.transform(X)  # distance to each centroid
        label = self.km_.predict(X).reshape(-1, 1)
        return np.column_stack([dists, label])

# Compare logistic regression with and without cluster features on breast cancer
data = load_breast_cancer()
Xbc, ybc = data.data, data.target

pipe_base = Pipeline([('scale', StandardScaler()), ('lr', LogisticRegression(max_iter=500))])

pipe_cluster = Pipeline([
    ('scale', StandardScaler()),
    ('kmf', KMeansFeatures(n_clusters=8)),
    ('lr', LogisticRegression(max_iter=500))
])

from sklearn.model_selection import cross_val_score
for label, pipe in [('Baseline LR', pipe_base), ('LR + KMeans features', pipe_cluster)]:
    scores = cross_val_score(pipe, Xbc, ybc, cv=5, scoring='roc_auc')
    print(f"{label:25s} | AUC = {scores.mean():.4f} ± {scores.std():.4f}")

## Summary

| Method | Type | Key Assumption | Hyperparameters | Use Case |
|--------|------|---------------|-----------------|----------|
| PCA | Linear dim-reduction | Variance = signal | n_components | Preprocessing linear models; visualization; noise removal |
| t-SNE | Non-linear visualization | Preserve local structure | perplexity | 2D/3D visualization only; not for ML preprocessing |
| UMAP | Non-linear dim-reduction | Manifold structure | n_neighbors, min_dist | Visualization AND preprocessing; faster than t-SNE |
| K-Means | Partition clustering | Spherical, equal-size clusters | K, n_init | Customer segmentation; cluster features; fast |
| DBSCAN | Density clustering | Dense connected regions | eps, min_samples | Arbitrary shapes; outlier detection; no K needed |

| Evaluation Metric | What It Measures | Range |
|-------------------|-----------------|-------|
| Inertia (WCSS) | Within-cluster compactness | 0 → ∞ (lower = better) |
| Silhouette score | Cluster separation vs. compactness | [-1, 1] (higher = better) |
| Explained variance ratio | PCA information retained | [0, 1] (higher = more info) |